# Project 1: Data Cleaning & Preparation
**Program:** DecodeLabs Data Analytics Internship  
**Dataset:** Order Records (1,200 rows × 14 columns)  
**Tool:** Python (pandas, openpyxl)

## Objective
Clean and prepare a raw order records dataset by handling missing values, 
removing duplicates, and standardizing formats across all columns. 
The final output is a cleaned Excel file and a documented change log.

In [3]:
import pandas as pd

## Step 1: Loading the Dataset

Before touching anything, we load the raw data and run a full diagnostic. 
The goal here is to understand the shape of the data, spot missing values, 
and identify anything that needs fixing before we start cleaning.

In [4]:
df = pd.read_excel('Dataset for Data Analytics.xlsx')

print("Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nData Types & Non-Null Counts:")
print(df.info())
print("\nMissing Values Per Column:")
print(df.isnull().sum())

Shape: (1200, 14)

Column Names:
['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice', 'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'TrackingNumber', 'ItemsInCart', 'CouponCode', 'ReferralSource', 'TotalPrice']

Data Types & Non-Null Counts:
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   str           
 1   Date             1200 non-null   datetime64[us]
 2   CustomerID       1200 non-null   str           
 3   Product          1200 non-null   str           
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   str           
 7   PaymentMethod    1200 non-null   str           
 8   OrderStatus      1200 non-null   str           
 9   TrackingNumber   1200 non-null   str           
 10 

## Phase 1: Handling Missing Values

The only column with missing values is CouponCode (309 nulls).
A null here doesn't mean the data is broken, it means the customer 
simply didn't apply a coupon. We fill with "NO_COUPON" to make that 
explicit rather than leaving a blank gap in the data.

In [5]:
df['CouponCode'] = df['CouponCode'].fillna('NO_COUPON')

print("Missing values after fix:")
print(df.isnull().sum())

Missing values after fix:
OrderID            0
Date               0
CustomerID         0
Product            0
Quantity           0
UnitPrice          0
ShippingAddress    0
PaymentMethod      0
OrderStatus        0
TrackingNumber     0
ItemsInCart        0
CouponCode         0
ReferralSource     0
TotalPrice         0
dtype: int64


### Result
All 309 missing CouponCode values have been filled with "NO_COUPON". 
The dataset now has zero null values across all 14 columns. 
No rows were deleted, all 1,200 records are preserved.

## Phase 2: Checking for Duplicates

Every order should appear exactly once in the dataset. 
Duplicate records inflate transaction counts and corrupt any 
analysis built on top of this data. We check for two things:
full duplicate rows, and duplicate OrderIDs specifically.

In [6]:
full_duplicates = df.duplicated().sum()
order_id_duplicates = df.duplicated(subset=['OrderID']).sum()

print("Fully duplicate rows:", full_duplicates)
print("Duplicate OrderIDs:", order_id_duplicates)

Fully duplicate rows: 0
Duplicate OrderIDs: 0


### Result
No duplicate rows or duplicate OrderIDs were found. 
All 1,200 records are unique. No action was required for this phase.

## Phase 3: Standardizing Formats

### 3a: Date Format
The Date column is currently stored as a datetime object. 
We need to convert it to a clean ISO 8601 string format (YYYY-MM-DD) 
which is the international standard for dates and what the project requires.

In [7]:
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')

print("Sample dates after formatting:")
print(df['Date'].head(10))

Sample dates after formatting:
0    2023-01-04
1    2024-08-23
2    2024-02-27
3    2023-10-15
4    2025-05-08
5    2023-10-23
6    2025-06-17
7    2023-05-12
8    2025-04-02
9    2023-11-21
Name: Date, dtype: str


### 3b: Whitespace Trimming
Invisible leading or trailing whitespace in text columns can cause 
silent errors, for example "Pending" and "Pending " would be treated 
as two different values in any grouping or filtering operation. 
We trim all string columns as a standard defensive measure.

In [8]:
text_columns = ['OrderID', 'CustomerID', 'Product', 'ShippingAddress', 
                'PaymentMethod', 'OrderStatus', 'TrackingNumber', 
                'CouponCode', 'ReferralSource']

for column in text_columns:
    df[column] = df[column].str.strip()

print("Whitespace trimming applied to all text columns.")

Whitespace trimming applied to all text columns.


### 3c: Numeric Precision
Monetary columns should always be rounded to 2 decimal places. 
Floating point arithmetic can sometimes introduce tiny rounding 
errors like 199.999999 instead of 200.00. We enforce 2 decimal 
places on UnitPrice and TotalPrice as a standard measure.

In [9]:
df['UnitPrice'] = df['UnitPrice'].round(2)
df['TotalPrice'] = df['TotalPrice'].round(2)

print("Sample UnitPrice values:")
print(df['UnitPrice'].head(10))
print("\nSample TotalPrice values:")
print(df['TotalPrice'].head(10))

Sample UnitPrice values:
0    570.62
1    151.35
2    550.68
3    273.19
4    626.01
5    245.86
6    664.42
7    149.55
8    134.28
9    509.38
Name: UnitPrice, dtype: float64

Sample TotalPrice values:
0    2853.10
1     302.70
2    2753.40
3     273.19
4    2504.04
5     491.72
6     664.42
7     747.75
8     268.56
9    2037.52
Name: TotalPrice, dtype: float64


### Result
All three formatting standards have been applied:
- Date column converted to ISO 8601 format (YYYY-MM-DD)
- All 9 text columns trimmed of leading and trailing whitespace
- UnitPrice and TotalPrice confirmed at 2 decimal places

The dataset is now fully cleaned and ready for export.

## Final Verification

Before exporting, we run a final check to confirm the dataset meets 
all project requirements. Zero nulls, zero duplicates, and correct 
date formatting.

In [10]:
print("Final Shape:", df.shape)
print("\nNull values remaining:")
print(df.isnull().sum())
print("\nDuplicate OrderIDs:", df.duplicated(subset=['OrderID']).sum())
print("\nSample of cleaned data:")
print(df.head(10))

Final Shape: (1200, 14)

Null values remaining:
OrderID            0
Date               0
CustomerID         0
Product            0
Quantity           0
UnitPrice          0
ShippingAddress    0
PaymentMethod      0
OrderStatus        0
TrackingNumber     0
ItemsInCart        0
CouponCode         0
ReferralSource     0
TotalPrice         0
dtype: int64

Duplicate OrderIDs: 0

Sample of cleaned data:
     OrderID        Date CustomerID  Product  Quantity  UnitPrice  \
0  ORD200000  2023-01-04     C72649  Monitor         5     570.62   
1  ORD200001  2024-08-23     C75739    Phone         2     151.35   
2  ORD200002  2024-02-27     C81728   Tablet         5     550.68   
3  ORD200003  2023-10-15     C33540    Chair         1     273.19   
4  ORD200004  2025-05-08     C81840  Printer         4     626.01   
5  ORD200005  2023-10-23     C37249    Phone         2     245.86   
6  ORD200006  2025-06-17     C83492   Laptop         1     664.42   
7  ORD200007  2023-05-12     C41460  Monitor 

## Exporting the Cleaned Dataset

All cleaning phases are complete. We export the final dataset 
as an Excel file ready for analysis.

In [11]:
df.to_excel('Cleaned_Orders_Dataset.xlsx', index=False)

print("Cleaned dataset exported successfully.")

Cleaned dataset exported successfully.
